In [ ]:
# Downgrade scikit-learn to 1.6.1
!pip install -q scikit-learn==1.6.1

In [ ]:
import sklearn
print(sklearn.__version__)  # should show 1.6.1

1.6.1


In [ ]:
!python --version

Python 3.11.13


In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder
from sklearn.compose import ColumnTransformer

from sklearn.svm import SVR

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error

from sklearn.decomposition import PCA

In [ ]:
df = pd.read_csv("/content/gurgaon_properties_post_feature_selection_v2.csv")
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,flat,sector 36,0.82,3.0,2.0,2,New Property,850.0,0.0,0.0,0.0,Low,Low Floor
1,flat,sector 89,0.95,2.0,2.0,2,New Property,1226.0,1.0,0.0,0.0,Low,Mid Floor
2,flat,sohna road,0.32,2.0,2.0,1,New Property,1000.0,0.0,0.0,0.0,Low,High Floor
3,flat,sector 92,1.60,3.0,4.0,3+,Relatively New,1615.0,1.0,0.0,1.0,High,Mid Floor
4,flat,sector 102,0.48,2.0,2.0,1,Relatively New,582.0,0.0,1.0,0.0,High,Mid Floor


Now i revert back the changes, the changes we made in the feature selection i.e converting categorical text col to number. why coz i'll make a pipeline sort of thing in which i will implement this conversion, after this we won't be

In [ ]:
df['furnishing_type'].value_counts()

,count
furnishing_type,
0.0,2349
1.0,1018
2.0,187


In [ ]:
#  0 -> unfurnished, 1 -> semi 2-> furnished

df['furnishing_type'] = df['furnishing_type'].replace({0.0:'unfurnished',1.0:'semifurnished',2.0:'furnished'})

In [ ]:
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,flat,sector 36,0.82,3.0,2.0,2,New Property,850.0,0.0,0.0,unfurnished,Low,Low Floor
1,flat,sector 89,0.95,2.0,2.0,2,New Property,1226.0,1.0,0.0,unfurnished,Low,Mid Floor
2,flat,sohna road,0.32,2.0,2.0,1,New Property,1000.0,0.0,0.0,unfurnished,Low,High Floor
3,flat,sector 92,1.60,3.0,4.0,3+,Relatively New,1615.0,1.0,0.0,semifurnished,High,Mid Floor
4,flat,sector 102,0.48,2.0,2.0,1,Relatively New,582.0,0.0,1.0,unfurnished,High,Mid Floor


In [ ]:
X = df.drop(columns=['price'])
y = df['price']

In [ ]:
y_transformed = np.log1p(y) # coz right skewed hai

Scene hai, pipeline mai i'll pass the encoding for diffrent col along with model and calculate accuray

## Ordinal Encoding

In [ ]:
columns_to_encode = ['property_type','sector','balcony','agePossession','furnishing_type','luxury_category','floor_category']


In [ ]:
preprocessor = ColumnTransformer(
    transformers = [
        ('num',StandardScaler(), ['bedRoom','bathroom','built_up_area','servant room','store room']),
        ('cat',OrdinalEncoder(), columns_to_encode)
    ],
    remainder='passthrough'
)

In [ ]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [ ]:
print(type(X))
print(X.columns)

<class 'pandas.core.frame.DataFrame'>
Index(['property_type', 'sector', 'bedRoom', 'bathroom', 'balcony',
       'agePossession', 'built_up_area', 'servant room', 'store room',
       'furnishing_type', 'luxury_category', 'floor_category'],
      dtype='object')


In [ ]:
print(len(X),len(y_transformed))

3554 3554


In [ ]:
kfold = KFold(n_splits=10, shuffle=True,random_state=42)
scores = cross_val_score(pipeline,X,y_transformed,cv=kfold,scoring='r2')

In [ ]:
scores.mean(),scores.std()

(np.float64(0.7363096633436828), np.float64(0.0323800575442993))

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y_transformed,test_size = 0.2,random_state = 42)

In [ ]:
pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['bedRoom', 'bathroom',
                                                   'built_up_area',
                                                   'servant room',
                                                   'store room']),
                                                 ('cat', OrdinalEncoder(),
                                                  ['property_type', 'sector',
                                                   'balcony', 'agePossession',
                                                   'furnishing_type',
                                                   'luxury_category',
                                                   'floor_category'])])),
                ('regressor', LinearRegression())])

In [ ]:
y_pred = pipeline.predict(X_test)

In [ ]:
y_pred = np.expm1(y_pred)


# It is typically used when your original target variable was transformed using log1p (i.e., np.log1p(y)), which is log(1 + y).
# so if you had done : y_transformed = np.log1p(y) during training then you should do y_pred = np.expm1(y_pred) to bring predictions back to the original scale.

In [ ]:
mean_absolute_error(np.expm1(y_test),y_pred)

0.9463822160089357

# putting everthing in a function

In [ ]:
def scorer(model_name, model):

    output = []

    output.append(model_name)

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])

    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

    output.append(scores.mean())

    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

    pipeline.fit(X_train,y_train)

    y_pred = pipeline.predict(X_test)

    y_pred = np.expm1(y_pred)

    output.append(mean_absolute_error(np.expm1(y_test),y_pred))

    return output

In [ ]:
!pip install xgboost

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.neural_network import MLPRegressor
from xgboost import XGBRegressor  # Make sure xgboost is installed

In [ ]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(max_iter=500),
    'xgboost':XGBRegressor()
}

In [ ]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [ ]:
model_output
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])
model_df.sort_values(['mae'])

,name,r2,mae
10,xgboost,0.889488,0.504048
5,random forest,0.881377,0.532754
6,extra trees,0.869360,0.560333
7,gradient boosting,0.872626,0.575613
9,mlp,0.804734,0.701157
4,decision tree,0.777300,0.711534
8,adaboost,0.754337,0.816449
1,svr,0.764202,0.847264
2,ridge,0.736313,0.946339
0,linear_reg,0.736310,0.946382


Problem with ordinal encoding is, for category it induces an order by which our linear regression is working
Solution - try OneHotEncoding

# One Hot Encoding

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first'),['sector','agePossession','furnishing_type'])
    ],
    remainder='passthrough'
)

In [ ]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', LinearRegression())
])

In [ ]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [ ]:
scores.mean()

np.float64(0.8546068948092419)

In [ ]:
scores.std()

np.float64(0.016000392991867098)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

In [ ]:
pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['bedRoom', 'bathroom',
                                                   'built_up_area',
                                                   'servant room',
                                                   'store room']),
                                                 ('cat', OrdinalEncoder(),
                                                  ['property_type', 'sector',
                                                   'balcony', 'agePossession',
                                                   'furnishing_type',
                                                   'luxury_category',
                                                   'floor_category']),
                                                 ('cat1',
                                                  OneHotEncoder(drop='first'),
                                                  ['sector', 'agePossession',
                                                   'furnishing_type'])])),
                ('regressor', LinearRegression())])

In [ ]:
y_pred = pipeline.predict(X_test)

In [ ]:
y_pred = np.expm1(y_pred)

In [ ]:
mean_absolute_error(np.expm1(y_test),y_pred)

0.6497630299214462

In [ ]:
def scorer(model_name, model):

    output = []

    output.append(model_name)

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])

    # K-fold cross-validation
    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

    output.append(scores.mean())

    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

    pipeline.fit(X_train,y_train)

    y_pred = pipeline.predict(X_test)

    y_pred = np.expm1(y_pred)

    output.append(mean_absolute_error(np.expm1(y_test),y_pred))

    return output

In [ ]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [ ]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [ ]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [ ]:
model_df.sort_values(['mae'])

,name,r2,mae
6,extra trees,0.894374,0.467687
10,xgboost,0.895850,0.493456
5,random forest,0.891312,0.497350
9,mlp,0.872171,0.521376
7,gradient boosting,0.876550,0.569293
0,linear_reg,0.854607,0.649763
2,ridge,0.854645,0.652985
4,decision tree,0.809037,0.693916
1,svr,0.769752,0.834124
8,adaboost,0.753424,0.843702


OneHotEncoding with PCA

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False),['sector','agePossession'])
    ],
    remainder='passthrough'
)

In [ ]:
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('pca', PCA(n_components=0.95)),
    ('regressor', LinearRegression())
])

In [ ]:
kfold = KFold(n_splits=10, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

In [ ]:
scores.mean()

np.float64(0.06225201431451136)

In [ ]:
scores.std()

np.float64(0.019860594071640172)

In [ ]:
def scorer(model_name, model):

    output = []

    output.append(model_name)

    pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('pca', PCA(n_components=0.95)),
        ('regressor', model)
    ])

    kfold = KFold(n_splits=10, shuffle=True, random_state=42)
    scores = cross_val_score(pipeline, X, y_transformed, cv=kfold, scoring='r2')

    output.append(scores.mean())

    X_train, X_test, y_train, y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

    pipeline.fit(X_train,y_train)

    y_pred = pipeline.predict(X_test)

    y_pred = np.expm1(y_pred)

    output.append(mean_absolute_error(np.expm1(y_test),y_pred))

    return output

In [ ]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [ ]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [ ]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])

In [ ]:
model_df.sort_values(['mae'])

,name,r2,mae
5,random forest,0.760819,0.661842
6,extra trees,0.739086,0.691417
4,decision tree,0.696442,0.761509
10,xgboost,0.622205,0.967581
7,gradient boosting,0.610623,0.987906
1,svr,0.218087,1.361143
9,mlp,0.206718,1.427022
8,adaboost,0.307379,1.446927
2,ridge,0.062252,1.526707
0,linear_reg,0.062252,1.526707


# Target Encoder

Conclusion : Using PCA, all models r2 and mae score comes down,now what --- we can try for target encoding
Major issue in target encoding "data leakage"

In [ ]:
!pip install category_encoders

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.7/85.7 kB 4.0 MB/s eta 0:00:00


In [ ]:
import category_encoders as ce

In [ ]:
preprocessor = ColumnTransformer(
    transformers = [
        ('num',StandardScaler(),['bedRoom','bathroom','built_up_area','servant room','store room']),
        ('cat',OrdinalEncoder(),columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False),['agePossession']),
        ('target_enc',ce.TargetEncoder(),['sector'])
    ],
    remainder='passthrough'
)

In [ ]:
pipeline = Pipeline([
    ('preprocessor',preprocessor),
    ('regressor',LinearRegression())
])

In [ ]:
kfold = KFold(n_splits=10,shuffle=True,random_state=42)
scores = cross_val_score(pipeline,X,y_transformed,cv=kfold,scoring='r2')

In [ ]:
scores.mean(),scores.std()

(np.float64(0.829521918225536), np.float64(0.018384463379122886))

In [ ]:
def scorer(model_name, model):

  output = []

  output.append(model_name)

  pipeline = Pipeline([
      ('preprocessor',preprocessor),
      ('regressor', model)
  ])

  kfold = KFold(n_splits=10,shuffle=True,random_state=42)
  scores = cross_val_score(pipeline,X,y_transformed,cv=kfold,scoring='r2')

  output.append(scores.mean())

  X_train,X_test,y_train,y_test = train_test_split(X,y_transformed,test_size=0.2,random_state=42)

  pipeline.fit(X_train,y_train)
  y_pred = pipeline.predict(X_test)
  y_pred = np.expm1(y_pred)

  output.append(mean_absolute_error(np.expm1(y_test),y_pred))

  return output

In [ ]:
model_dict = {
    'linear_reg':LinearRegression(),
    'svr':SVR(),
    'ridge':Ridge(),
    'LASSO':Lasso(),
    'decision tree': DecisionTreeRegressor(),
    'random forest':RandomForestRegressor(),
    'extra trees': ExtraTreesRegressor(),
    'gradient boosting': GradientBoostingRegressor(),
    'adaboost': AdaBoostRegressor(),
    'mlp': MLPRegressor(),
    'xgboost':XGBRegressor()
}

In [ ]:
model_output = []
for model_name,model in model_dict.items():
    model_output.append(scorer(model_name, model))

In [ ]:
model_df = pd.DataFrame(model_output, columns=['name','r2','mae'])
model_df.sort_values(['mae'])

,name,r2,mae
10,xgboost,0.904798,0.447518
6,extra trees,0.901876,0.454961
5,random forest,0.899761,0.466013
7,gradient boosting,0.889125,0.506611
4,decision tree,0.830733,0.573440
9,mlp,0.852189,0.589689
8,adaboost,0.815143,0.689550
0,linear_reg,0.829522,0.713011
2,ridge,0.829536,0.713523
1,svr,0.782917,0.818851


### Conclusion : random forest and XGboost are best,  xgboost with bayesian hyper parameter are best

## Hyperparameter Tuning :  Random Forest

In [ ]:
from sklearn.model_selection import GridSearchCV

In [ ]:
param_grid = {
    'regressor__n_estimators' : [300],
    'regressor__max_depth' : [20],
    'regressor__max_samples' : [1.0],
    'regressor__max_features' : ['sqrt']
}

In [ ]:
df.head()

,property_type,sector,price,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,flat,sector 36,0.82,3.0,2.0,2,New Property,850.0,0.0,0.0,unfurnished,Low,Low Floor
1,flat,sector 89,0.95,2.0,2.0,2,New Property,1226.0,1.0,0.0,unfurnished,Low,Mid Floor
2,flat,sohna road,0.32,2.0,2.0,1,New Property,1000.0,0.0,0.0,unfurnished,Low,High Floor
3,flat,sector 92,1.60,3.0,4.0,3+,Relatively New,1615.0,1.0,0.0,semifurnished,High,Mid Floor
4,flat,sector 102,0.48,2.0,2.0,1,Relatively New,582.0,0.0,1.0,unfurnished,High,Mid Floor


In [ ]:
columns_to_encode = ['property_type','sector', 'balcony', 'agePossession', 'furnishing_type', 'luxury_category', 'floor_category']

# Creating a column transformer for preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), ['bedRoom', 'bathroom', 'built_up_area', 'servant room', 'store room']),
        ('cat', OrdinalEncoder(), columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False),['agePossession']),
        ('target_enc', ce.TargetEncoder(), ['sector'])
    ],
    remainder='passthrough'
)

In [ ]:
pipeline = Pipeline([
    ('preprocessor' , preprocessor),
    ('regressor', RandomForestRegressor())
])

In [ ]:
kfold = KFold(n_splits=10,shuffle=True,random_state=42)

In [ ]:
search = GridSearchCV(pipeline,param_grid,cv=kfold,scoring='r2',n_jobs=1,verbose=4)

In [ ]:
search.fit(X,y_transformed)

Fitting 10 folds for each of 1 candidates, totalling 10 fits
[CV 1/10] END regressor__max_depth=20, regressor__max_features=sqrt, regressor__max_samples=1.0, regressor__n_estimators=300;, score=0.912 total time=   1.7s
[CV 2/10] END regressor__max_depth=20, regressor__max_features=sqrt, regressor__max_samples=1.0, regressor__n_estimators=300;, score=0.930 total time=   1.9s
[CV 3/10] END regressor__max_depth=20, regressor__max_features=sqrt, regressor__max_samples=1.0, regressor__n_estimators=300;, score=0.911 total time=   1.9s
[CV 4/10] END regressor__max_depth=20, regressor__max_features=sqrt, regressor__max_samples=1.0, regressor__n_estimators=300;, score=0.895 total time=   1.6s
[CV 5/10] END regressor__max_depth=20, regressor__max_features=sqrt, regressor__max_samples=1.0, regressor__n_estimators=300;, score=0.908 total time=   1.6s
[CV 6/10] END regressor__max_depth=20, regressor__max_features=sqrt, regressor__max_samples=1.0, regressor__n_estimators=300;, score=0.894 total time

GridSearchCV(cv=KFold(n_splits=10, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('preprocessor',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('num',
                                                                         StandardScaler(),
                                                                         ['bedRoom',
                                                                          'bathroom',
                                                                          'built_up_area',
                                                                          'servant '
                                                                          'room',
                                                                          'store '
                                                                          'room']),
                                                                        ('cat',
                                                                         OrdinalEncoder(),
                                                                         ['property_type',
                                                                          'sector',
                                                                          'balcony',
                                                                          'agePossession',
                                                                          'furnishing_type',...
                                                                          'floor_category']),
                                                                        ('cat1',
                                                                         OneHotEncoder(drop='first',
                                                                                       sparse_output=False),
                                                                         ['agePossession']),
                                                                        ('target_enc',
                                                                         TargetEncoder(),
                                                                         ['sector'])])),
                                       ('regressor', RandomForestRegressor())]),
             n_jobs=1,
             param_grid={'regressor__max_depth': [20],
                         'regressor__max_features': ['sqrt'],
                         'regressor__max_samples': [1.0],
                         'regressor__n_estimators': [300]},
             scoring='r2', verbose=4)

In [ ]:
final_pipe =search.best_estimator_

In [ ]:
search.best_params_

{'regressor__max_depth': 20,
 'regressor__max_features': 'sqrt',
 'regressor__max_samples': 1.0,
 'regressor__n_estimators': 300}

In [ ]:
search.best_score_

np.float64(0.901839926351203)

In [ ]:
final_pipe.fit(X,y_transformed)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['bedRoom', 'bathroom',
                                                   'built_up_area',
                                                   'servant room',
                                                   'store room']),
                                                 ('cat', OrdinalEncoder(),
                                                  ['property_type', 'sector',
                                                   'balcony', 'agePossession',
                                                   'furnishing_type',
                                                   'luxury_category',
                                                   'floor_category']),
                                                 ('cat1',
                                                  OneHotEncoder(drop='first',
                                                                sparse_output=False),
                                                  ['agePossession']),
                                                 ('target_enc', TargetEncoder(),
                                                  ['sector'])])),
                ('regressor',
                 RandomForestRegressor(max_depth=20, max_features='sqrt',
                                       max_samples=1.0, n_estimators=300))])

### Export model


In [ ]:
preprocessor = ColumnTransformer(
    transformers = [
        ('num',StandardScaler(),['bedRoom','bathroom','built_up_area','servant room','store room']),
        ('cat',OrdinalEncoder(),columns_to_encode),
        ('cat1',OneHotEncoder(drop='first',sparse_output=False),['sector','agePossession'])
    ],
    remainder = 'passthrough'
)

In [ ]:
pipeline.fit(X,y_transformed)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['bedRoom', 'bathroom',
                                                   'built_up_area',
                                                   'servant room',
                                                   'store room']),
                                                 ('cat', OrdinalEncoder(),
                                                  ['property_type', 'sector',
                                                   'balcony', 'agePossession',
                                                   'furnishing_type',
                                                   'luxury_category',
                                                   'floor_category']),
                                                 ('cat1',
                                                  OneHotEncoder(drop='first',
                                                                sparse_output=False),
                                                  ['agePossession']),
                                                 ('target_enc', TargetEncoder(),
                                                  ['sector'])])),
                ('regressor', RandomForestRegressor())])

In [ ]:
import pickle

with open('pipeline.pkl','wb') as file:
  pickle.dump(pipeline,file)

In [ ]:
with open('df.pkl','wb') as file:
  pickle.dump(X,file)

# Target to get mean error under 25
##### how ?
1. More Data
2. Xgboost + hyperparameter
3. Revisit the entire process

In [ ]:
data = [['house', 'sector 63', 4, 4, '2', 'New Property', 2042, 0, 0, 'furnished','High', 'Low Floor']]
columns = ['property_type','sector','bedRoom', 'bathroom', 'balcony','agePossession','built_up_area', 'servant room', 'store room','furnishing_type', 'luxury_category', 'floor_category']

one_df = pd.DataFrame(data,columns=columns)
one_df

,property_type,sector,bedRoom,bathroom,balcony,agePossession,built_up_area,servant room,store room,furnishing_type,luxury_category,floor_category
0,house,sector 63,4,4,2,New Property,2042,0,0,furnished,High,Low Floor


In [ ]:
np.expm1(pipeline.predict(one_df))

array([2.93316282])

In [ ]:
# ✅ Install joblib if not already installed
!pip install joblib

import joblib

# 🔁 Save your trained model (assumes it's named 'pipeline')
joblib.dump(pipeline, 'pipeline1.pkl')

# 📥 Download the file to your local machine
from google.colab import files
files.download('pipeline1.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import joblib
joblib.dump(pipeline, 'pipeline_compatible.joblib')

# Download:
from google.colab import files
files.download('pipeline_compatible.joblib')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>